# Lab 02: The Tool Use Pattern for Evidence Inspection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab02_tool_use_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

## What this lab is about

In Lab 01 the AI model reviewed its own writing. Here it does something different. Instead of
answering from whatever text we give it, the AI can **ask to use tools**: small programs that
read the real evidence files and report back what they find. The AI then reasons over those
results. This is called the **Tool Use Pattern**.

One important point: the AI never opens the evidence files itself. It can only **ask** for a
tool to be run. Our code runs the tool and decides exactly what the AI gets to see. That keeps
every answer tied to something you can check.

You will answer one case question: *does a recovered phone contain evidence that a stolen black
SUV was photographed and prepared for an online sale?* You will answer it twice:

- **Part A:** you **call the tools yourself**, read the results, and then ask the AI to write a
  report from them.
- **Part B:** you hand the same tools to a **`ToolAgent`**, a helper that lets the AI choose
  which tools to call on its own.

The agent is a tool-use aid; the forensic judgment stays with you.

## How to use this notebook

- A notebook is made of **cells**. Text cells (like this one) explain. Code cells (the grey
  boxes) do the work.
- Run the code cells **one at a time, from top to bottom**. Click a cell and press
  **Shift + Enter**, or click its Run button. Wait for each cell to finish before the next.
- Every code cell has a text cell just before it with four parts: **Purpose** (why the step
  exists), **Input** (what it uses, and which earlier step made it), **How it works** (what
  the code does, in plain words), and **Output** (what you will see, and what the next step
  uses).
- You do not need to know Python. The lines starting with `#` inside the code cells are
  comments: plain-English notes that explain what each part does. The computer ignores them.
- If the session restarts (for example after a long break), run the notebook again from the
  top.

## The plan at a glance

Each step hands something to the next one. This table shows the chain.

| Step | What happens | What it hands on |
|---|---|---|
| 0.1 to 0.4 | Setup: install one add-on, connect to the AI model, write the case files, load the tool framework | `client`, `MODEL`, the `data/` folder, `tool`, `ToolAgent` |
| A.1 | Load the evidence and define three tools that read it | `forensic_tools` |
| A.2 | Look at the description card each tool shows the AI | (a preview only) |
| A.3 | Run the three tools yourself | `manual_tool_log`, `manual_evidence_bundle` |
| A.4 | The AI writes a report from the evidence you gathered | `manual_report` |
| B.1 | Build the agent and write its task | `tool_agent`, `agent_user_msg` |
| B.2 | The agent picks and runs the tools, then writes a report | `agent_report` |
| Your Evidence Report | You write the forensic conclusion | your `report` |
| Compare the two workflows | You judge the agent against your own work | your written answers |

## Part 0: One-Time Setup

These four steps get the notebook ready. They install one small add-on, connect the notebook to
a **free AI model** that runs on the internet (on a service called Groq), create the case files,
and load the small framework that lets the AI use tools. Everything runs in this hosted
session; nothing is installed on your own computer.

### Step 0.1: Install the add-on that talks to the AI model

**Purpose:** add the one tool this notebook needs to send messages to an AI model and read its
replies.

**Input:** nothing from you. This is the first step.

**How it works:** the command `%pip install` downloads a small, free Python add-on (called a
**library**) named `openai` into this hosted session. Despite its name, it can talk to several
AI services, including Groq, which we use here.

**Output:** a few lines of install messages, or none at all. Warnings are normal and safe to
ignore. The add-on is now ready for Step 0.2.

In [ ]:
# Step 0.1: install the one add-on this notebook needs, inside this hosted session only.
# The -q means "quiet": show fewer messages.
%pip install -q openai

### Step 0.2: Connect to a free AI model (Groq)

**Purpose:** connect the notebook to a free AI model, so the later steps can send it evidence
and get written answers back.

**Input:** the `openai` add-on from Step 0.1, and a free Groq **API key**. An API key is a
private password that lets the notebook use the AI model. To get one:

1. Open **https://console.groq.com/keys**, sign in, and click **Create API Key**. Copy the key.
2. Run the cell below and paste the key when asked. It stays hidden while you type and is
   **not saved** into the notebook. Treat it like a password: do not share it.

**How it works:** the code keeps your key for this session only and connects to Groq. Then it
picks the first AI model on its short list that your key can use. In this lab a model called
Qwen comes first, because Part B needs a model that reliably writes out its tool requests. The
cell also adds a small safety helper that keeps every request inside Groq's free limits, and
finally sends a one-word test message to prove the connection works.

**Output:** a line such as `Model OK: qwen/qwen3.8-27b -> ready`. The cell also creates two
things that every AI step below uses: `client` (the connection to Groq) and `MODEL` (the name
of the AI model it chose).

In [ ]:
# Step 0.2: connect this notebook to a free AI model hosted by Groq.
import os        # os: lets the notebook read and store settings, such as your key
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: sends your messages to the AI model and brings back replies
                          # (Groq understands the same message format)

# Ask for your key only if this session does not already have one stored.
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

# The web address of Groq's service. Every message to the AI model is sent here.
BASE_URL = 'https://api.groq.com/openai/v1'
# A short list of free AI models that work well for this lab, best first. Groq retires models
# from time to time, so this cell uses the first one your key can reach and prints its name.
# You can edit this list to try a different model.
# Qwen comes first here: in Part B the agent must write its tool requests as plain text inside
# <tool_call> tags. Qwen does this reliably; gpt-oss tends to plan the request silently and
# send back an empty reply.
PREFERRED = ['qwen/qwen3.8-27b', 'openai/gpt-oss-120b']

# Build the 'client': the connection that every later step uses to talk to the AI model.
# The free plan only allows a certain amount of text per minute (about 8,000 tokens; a token is
# a small piece of text, roughly three-quarters of a word). If you go over, Groq says 'try again
# in a few seconds'. max_retries=8 tells the client to wait and try again instead of failing.
client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'], max_retries=8)

# Every AI request in this notebook passes through the small helper below. You do not need to
# follow every line. It does two things:
# 1. It caps how long each reply may be: 1,500 tokens, or 900 for the Qwen model, whose free
#    plan allows only 1,000 reply tokens a minute. Groq counts your request PLUS the longest
#    possible reply against your per-minute allowance, so a cap keeps you inside it.
#    (Any single request can still set its own max_tokens.)
# 2. The gpt-oss model sometimes tries to use a built-in feature instead of writing plain text,
#    and Groq refuses that ('model called a tool'). The helper then simply asks again, adding
#    a reminder to reply in plain text. It tries at most 3 times.
_create = client.chat.completions.create
def _create_within_limits(*args, **kw):
    kw.setdefault('max_tokens', 900 if 'qwen' in str(kw.get('model')) else 1500)
    for attempt in range(3):
        try:
            return _create(*args, **kw)
        except Exception as err:
            if 'called a tool' not in str(err) or attempt == 2:
                raise
            kw['messages'] = list(kw.get('messages', [])) + [
                {'role': 'user', 'content': 'Reply in plain text only; do not call functions.'}]
client.chat.completions.create = _create_within_limits

# Ask Groq which models your key can use, then pick the first one from PREFERRED that is there.
available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available), None)
# If none of our preferred models is available, list the ones that are, and stop here so you
# can put one of them into PREFERRED.
if MODEL is None:
    print('None of PREFERRED is on your key right now. Models your key can use:')
    for m in available:
        print('  -', m)
    raise SystemExit('Put one of these chat models first in PREFERRED and re-run this cell.')

# Send a tiny test message to prove the whole connection works.
probe = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
print('Model OK:', MODEL, '->', (probe.choices[0].message.content or '').strip())

### Step 0.3: Write the case files

**Purpose:** create the evidence for case `TU-2026-004` so you can examine it. The six files are
**synthetic**: made up for teaching, so no real person's data is involved. They are identical
to the copies kept in the course repository.

**Input:** nothing from earlier steps. The text of all six files is written inside the code
cell itself, so you do not need to download anything.

**How it works:** the code makes a folder called `data/` and writes each of the six files into
it. If a file is already there, it is left untouched.

**Output:** one line per file showing where it is (for example `wrote data/media_index.csv`).
The line says `wrote` even when the file was already there; your existing copy is still not
changed. The six files are:

| File | What it records |
|---|---|
| `artifact_manifest.json` | The case summary: case number, phone model, how and when the phone was copied, and the time window under investigation. |
| `media_index.csv` | The list of photos found on the phone: name, location on the phone, when it was created, and its size. |
| `image_metadata.csv` | Details saved inside each photo (its **metadata**): when it was taken, when it was edited, GPS location, and camera. |
| `vehicle_detections.csv` | Results of an automatic image check that labels the vehicle in each photo: colour, body type, roof rack, whether a number-plate area is visible, and how confident the check was (0 to 1). |
| `listing_drafts.json` | Saved drafts of online sale listings: title, description, attached photos, and when (if ever) each was published. |
| `chain_of_custody.csv` | Who handled the evidence and when: the **chain of custody**. |

A `.csv` file is a simple table saved as text, one row per line. A `.json` file holds
structured data: labelled fields and lists, in a standard text format called **JSON** that both
people and programs can read. Step A.1 reads these files.

In [ ]:
# Step 0.3: write the six case files into a folder called data/, so no download is needed.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

# ARTIFACTS (the long line below) is a dictionary: a list of name -> value pairs. Each name is a
# file name, and each value is the full text of that file. You do not need to read it here.
ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "TU-2026-004",\n  "lab": "Lab 02 - Tool Use Pattern",\n  "device": {\n    "platform": "Android",\n    "model": "Samsung Galaxy S23",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "logical_plus_media_export",\n    "tool": "Magnet AXIOM 8.1",\n    "acquired_at_utc": "2026-01-05T14:22:09Z",\n    "original_image_sha256": "8d8f7c95578d2678d12b178f2636d2b566bf1b74153f2f8f7852d1fe6b9f08c4"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-01-02T00:00:00Z",\n    "end": "2026-01-03T03:00:00Z"\n  },\n  "note": "Synthetic classroom dataset for tool-use-pattern demo"\n}\n', 'media_index.csv': 'file_name,path,created_at_utc,size_bytes,folder\nIMG_2044.jpg,/storage/emulated/0/DCIM/Camera/IMG_2044.jpg,2026-01-02T21:14:03Z,2841933,DCIM/Camera\nIMG_2045.jpg,/storage/emulated/0/DCIM/Camera/IMG_2045.jpg,2026-01-02T21:17:22Z,2738810,DCIM/Camera\nIMG_2051.jpg,/storage/emulated/0/DCIM/Camera/IMG_2051.jpg,2026-01-02T22:04:10Z,2911022,DCIM/Camera\n', 'image_metadata.csv': 'file_name,captured_at_utc,edited_at_utc,gps_lat,gps_lon,camera_model\nIMG_2044.jpg,2026-01-02T21:14:03Z,,40.728640,-73.794853,Galaxy S23 Rear Camera\nIMG_2045.jpg,2026-01-02T21:17:22Z,,40.728655,-73.794841,Galaxy S23 Rear Camera\nIMG_2051.jpg,2026-01-02T22:04:10Z,2026-01-02T22:10:44Z,40.728611,-73.794909,Galaxy S23 Rear Camera\n', 'vehicle_detections.csv': 'file_name,primary_object,color,body_type,roof_rack,plate_region_visible,confidence\nIMG_2044.jpg,vehicle,black,SUV,yes,yes,0.96\nIMG_2045.jpg,vehicle,silver,sedan,no,no,0.74\nIMG_2051.jpg,vehicle,black,SUV,no,yes,0.68\n', 'listing_drafts.json': '[\n  {\n    "draft_id": "draft-9001",\n    "created_at_utc": "2026-01-02T21:31:11Z",\n    "title": "black SUV for sale",\n    "description": "clean title, low miles, cash only",\n    "attached_images": ["IMG_2044.jpg"],\n    "published_at_utc": null\n  },\n  {\n    "draft_id": "draft-9002",\n    "created_at_utc": "2026-01-02T22:18:54Z",\n    "title": "spare car parts",\n    "description": "miscellaneous trim and lights",\n    "attached_images": ["IMG_2051.jpg"],\n    "published_at_utc": null\n  }\n]\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-01-05T14:22:09Z,Investigator_A,acquired_device,"Logical extraction and media export to encrypted evidence drive"\n2026-01-05T14:48:21Z,Investigator_A,computed_hash,"SHA-256 verified against extraction log"\n2026-01-06T09:12:44Z,Analyst_B,created_working_copy,"Exported gallery, metadata, and listing draft artifacts"\n'}

# Make the data/ folder (exist_ok=True: no error if it is already there).
data_dir = Path('data'); data_dir.mkdir(exist_ok=True)
# A loop: repeat the indented lines once for each file in ARTIFACTS.
# Write the file only if it does not exist yet (so nothing is overwritten), then print its place.
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))

### Step 0.4: Load the tool framework

**Purpose:** load the small framework that turns an ordinary piece of Python code (a
**function**: a named, reusable block of code that takes inputs and returns a result) into a
**tool** the AI can ask to use. It also loads the `ToolAgent` that runs the whole ask, run,
answer loop in Part B.

**Input:** nothing from earlier steps. All the code is inside the cell.

**How it works:** the cell only **defines** things; it does not run anything yet. You do not
need to read every line. The three pieces that matter are:

- `@tool`: a label you put above a function to turn it into a tool. It writes a short
  **description card** for the tool (its name, what it does, and the inputs it needs).
- `Tool`: the tool itself: the function plus its description card.
- `ToolAgent`: shows the AI the description cards of all its tools, reads the AI's requests to
  use a tool, runs the tool, sends the result back, and repeats until the AI gives its answer.

The framework also protects you from messy AI requests: it skips a request that is broken or
names a tool that does not exist, ignores inputs a tool does not have, and turns a tool error
into a message for the AI instead of stopping the notebook.

This is a self-contained copy adapted from The Neural Maze's agentic-patterns-course (MIT
License).

**Output:** nothing is printed. Step A.1 uses `@tool` to build the three forensic tools, and
Step B.1 uses `ToolAgent`.

In [ ]:
# Step 0.4: the tool-use framework (Tool, @tool, ToolAgent), a self-contained copy for this
# notebook. Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The
# Neural Maze.

# This is the toolbox behind @tool and ToolAgent. Skim the names; you do not need to follow every
# line. The two parts that matter most are tool() (which turns a function into a tool) and
# ToolAgent.run() at the very bottom (the ask -> run tool -> send result back loop).
import json
import re
from dataclasses import dataclass
from typing import Any, Callable


# ---------------------------------------------------------------- tag extraction
# The AI writes each tool request between <tool_call> and </tool_call> tags. These two pieces
# find those tags in the AI's reply and pull out what is written between them.
@dataclass
class TagContentResult:
    content: list
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """Extract everything enclosed by <tag>...</tag> (used to read <tool_call> blocks)."""
    # Some AI models repeat the closing tag at the end; tidy that up first.
    if tag == "tool_call":
        text = re.sub(r"(</tool_call>\s*){2,}$", "</tool_call>", text.strip())
    pattern = rf"<{tag}>(.*?)</{tag}>"
    matches = re.findall(pattern, text, re.DOTALL)
    return TagContentResult(content=[m.strip() for m in matches], found=bool(matches))


# ---------------------------------------------------------------- chat history
# build_prompt_structure: wrap a piece of text as one message with a role: 'system' (the job
# description), 'user' (a request or a tool result), or 'assistant' (something the AI said).
def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


# update_chat_history: add one new message to the end of a conversation.
def update_chat_history(history: list, msg: str, role: str):
    history.append(build_prompt_structure(prompt=msg, role=role))


# ChatHistory: a conversation (a list of messages). An optional size limit drops the oldest
# message when the list is full; -1 (the default, used here) means "no limit".
class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        super().__init__(messages or [])
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


# completions_create: send a conversation to the AI model and return the text of its reply.
def completions_create(client, messages, model) -> str:
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


# ---------------------------------------------------------------- tool wrapper
# get_fn_signature: write a tool's description card from the function itself: its name, the
# text in triple quotes at its top, and the names and types of its inputs.
def get_fn_signature(fn: Callable) -> dict:
    """Build a JSON-serializable signature (name, description, parameter types) from a function."""
    sig = {"name": fn.__name__, "description": fn.__doc__, "parameters": {"properties": {}}}
    schema = {k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"}
    sig["parameters"]["properties"] = schema
    return sig


# validate_arguments: check the inputs the AI asked for before a tool runs. Inputs the tool does
# not have are dropped, and each input is converted to the type the card asks for (for example
# the number 5 becomes the text "5" if the tool expects text).
def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    """Coerce each argument to the type declared in the tool signature; drop undeclared keys."""
    properties = tool_signature["parameters"]["properties"]
    type_mapping = {"int": int, "str": str, "bool": bool, "float": float}
    validated = {}
    for name, value in tool_call["arguments"].items():
        schema = properties.get(name)
        if schema is None:
            continue  # AI models sometimes invent extra inputs; drop them
        converter = type_mapping.get(schema.get("type"))
        if converter is None:
            validated[name] = value
            continue
        if not isinstance(value, converter):
            value = converter(value)
        validated[name] = value
    tool_call["arguments"] = validated
    return tool_call


# Tool: one tool = the function that does the work, plus its description card.
class Tool:
    """Wraps a callable plus its JSON signature so a model can describe and call it."""

    def __init__(self, name: str, fn: Callable, fn_signature: str):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    # run: call the function with the given inputs and return its result.
    def run(self, **kwargs):
        return self.fn(**kwargs)


# tool: the @tool label. Writing @tool above a function turns it into a Tool with its card.
def tool(fn: Callable) -> Tool:
    """Decorator: turn a plain function into a Tool (its name, docstring, and typed parameters)."""
    sig = get_fn_signature(fn)
    return Tool(name=sig["name"], fn=fn, fn_signature=json.dumps(sig))


# ---------------------------------------------------------------- the agent
# The standing instructions (the 'system' message) the agent gives the AI. They explain how to
# ask for a tool. The %s is replaced by the description cards of all the agent's tools.
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call>
XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>,  "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools:

<tools>
%s
</tools>

Each <tool_call> block must contain exactly one valid JSON object with the keys "name", "arguments", and "id".
Do not put commentary, explanations, or nested <tool_call> tags inside a <tool_call> block.

When you have enough information from the tools, answer the user's question directly in plain text and do not
return any more <tool_call> tags.
"""


class ToolAgent:
    """Runs a model that may request local tools via <tool_call> blocks, executes them, and loops
    until the model answers in plain text or a round limit is reached."""

    # Set up the agent: remember the AI connection, the model, and the tools (by name).
    def __init__(self, tools, model="openai/gpt-oss-120b", client=None):
        if client is None:
            raise ValueError("ToolAgent requires a `client` (chat.completions.create).")
        self.client = client
        self.model = model
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {t.name: t for t in self.tools}

    # add_tool_signatures: join all the tools' description cards into one block of text.
    def add_tool_signatures(self) -> str:
        return "".join(t.fn_signature for t in self.tools)

    # process_tool_calls: carry out the AI's tool requests. For each request: read it, skip it if
    # it is broken or names an unknown tool, check its inputs, run the tool, and print what
    # happened. The results ("observations") are collected and returned, one per request id.
    def process_tool_calls(self, tool_calls_content: list) -> dict:
        observations = {}
        fallback_id = 0
        for tool_call_str in tool_calls_content:
            for tool_call in self._parse_tool_call_candidates(tool_call_str):
                norm = self._normalize_tool_call_dict(tool_call)
                if norm is None:
                    print(f"\nSkipping malformed tool call payload: {tool_call}")
                    continue
                tool_name = norm["name"]
                t = self.tools_dict.get(tool_name)
                if t is None:
                    print(f"\nSkipping unknown tool request: {tool_name}")
                    continue
                fallback_id += 1
                norm.setdefault("id", fallback_id)
                print(f"\nUsing Tool: {tool_name}")
                try:
                    validated = validate_arguments(norm, json.loads(t.fn_signature))
                    print(f"Tool call: {validated}")
                    result = t.run(**validated["arguments"])
                except Exception as e:   # a wrong or missing argument becomes an observation, not a crash
                    validated, result = norm, f"Tool error: {e}"
                print(f"Tool result:\n{result}")
                observations[validated["id"]] = result
        return observations

    # _parse_tool_call_candidates: read the text of one tool request. AI replies are not always
    # tidy, so it tries three ways: the whole text as one request, requests nested inside more
    # <tool_call> tags, and finally any {...} blocks found anywhere in the text.
    @staticmethod
    def _parse_tool_call_candidates(tool_call_str: str) -> list:
        stripped = tool_call_str.strip()
        candidates = []
        try:
            parsed = json.loads(stripped)
            if isinstance(parsed, dict):
                return [parsed]
        except json.JSONDecodeError:
            pass
        nested = extract_tag_content(stripped, "tool_call")
        if nested.found:
            for n in nested.content:
                candidates.extend(ToolAgent._parse_tool_call_candidates(n))
            if candidates:
                return candidates
        decoder = json.JSONDecoder()
        cursor = 0
        while cursor < len(stripped):
            start = stripped.find("{", cursor)
            if start == -1:
                break
            try:
                parsed, end = decoder.raw_decode(stripped[start:])
                if isinstance(parsed, dict):
                    candidates.append(parsed)
                    cursor = start + end
                    continue
            except json.JSONDecodeError:
                pass
            cursor = start + 1
        return candidates

    # _normalize_tool_call_dict: make sure a request has a tool name and a set of inputs. If the
    # name is missing, it uses the first text value it finds. Returns None if it cannot fix it.
    @staticmethod
    def _normalize_tool_call_dict(tool_call):
        if not isinstance(tool_call, dict):
            return None
        norm = dict(tool_call)
        if "name" not in norm:
            for key, value in norm.items():
                if key in {"arguments", "id"}:
                    continue
                if isinstance(value, str):
                    norm["name"] = value
                    break
        if "name" not in norm or "arguments" not in norm:
            return None
        if not isinstance(norm["arguments"], dict):
            return None
        return norm

    # run: the whole loop. Ask the AI, run any tools it requests, send back the results, repeat.
    def run(self, user_msg: str) -> str:
        # Start the conversation: the standing instructions (with the tool cards), then the task.
        history = ChatHistory([
            build_prompt_structure(TOOL_SYSTEM_PROMPT % self.add_tool_signatures(), "system"),
            build_prompt_structure(user_msg, "user"),
        ])
        # The round limit: with 3 tools this is 6 rounds. (The +2 leaves room for the
        # evidence-check reminders below.)
        max_rounds = max(3, len(self.tools) + 1) + 2
        last_output = ""
        tools_used, pushbacks = 0, 0
        # A loop: repeat once per round, up to max_rounds times.
        for _ in range(max_rounds):
            # Ask the AI for its next move, and look for tool requests in its reply.
            last_output = completions_create(self.client, history, self.model)
            tool_calls = extract_tag_content(str(last_output), "tool_call")
            if not tool_calls.found:
                if tools_used == 0 and pushbacks < 2:
                    # Evidence check: an answer (or an empty reply) given before any tool has
                    # run cannot be based on the evidence, so ask for a tool call first.
                    pushbacks += 1
                    if str(last_output).strip():
                        update_chat_history(history, last_output, "assistant")
                    update_chat_history(history, "You have not called any tool yet. Call the tool that "
                                        "shows the evidence, written as a <tool_call> block in plain text.", "user")
                    continue
                # No tool request, and tools were already used: this is the AI's final answer.
                return last_output
            # The AI asked for tools: save its request, run the tools, and send back the results
            # as an "Observation" message, so the AI can read them in the next round.
            update_chat_history(history, last_output, "assistant")
            observations = self.process_tool_calls(tool_calls.content)
            tools_used += len(observations)
            update_chat_history(history, f"Observation: {json.dumps(observations)}", "user")
        # Out of rounds: ask the AI for its best final answer from the results it already has.
        update_chat_history(history,
                            "You have reached the tool-call limit. Provide the best final answer "
                            "from the available observations.", "user")
        return completions_create(self.client, history, self.model)

## The Case in One Minute

**Case `TU-2026-004`:** a black SUV with a roof rack is reported stolen on 2 January 2026. Police
recover a Samsung Galaxy S23 phone believed to belong to the seller. The phone holds a gallery
of photos, the photos' metadata, automatic vehicle detections, and saved sale-listing drafts.

**Your question:** does the phone contain **confirmed, likely, or unconfirmed** evidence that a
stolen black SUV was photographed and prepared for an online sale after 2 January 2026?

**Three tools read the evidence for you** (you build them in Step A.1):

| Tool | What it does |
|---|---|
| `list_media_files` | Finds the photos in a given folder that were created on or after a given time. |
| `inspect_image_evidence` | Looks at one photo from every angle (file details, metadata, vehicle detection) and compares it with the stolen vehicle's description. |
| `inspect_listing_records` | Finds saved sale-listing drafts created on or after a given time. |

The natural order is: find the photos (`list_media_files`), inspect the promising ones
(`inspect_image_evidence`), then check the sale drafts (`inspect_listing_records`).

All times are in **UTC**, a world-standard clock that forensic tools use so evidence from
different places lines up. They are written like `2026-01-02T21:14:03Z`: date, then `T`, then
the time, then `Z` for UTC.

## Part A: Call the Tools Yourself

First **you** act as the agent: pick each tool, give it sensible inputs, run it, and read the
result. Then you hand your results to the AI and ask it to write them up. Doing it by hand first
shows you what a good sequence of tool calls looks like, so you can judge the agent in Part B.

### Step A.1: Load the case records and build the three tools

**Purpose:** read the evidence into memory and build the three forensic tools that look things
up in it.

**Input:** the files in the `data/` folder from Step 0.3, and the `@tool` label from Step 0.4.

**How it works:** the code reads five of the six case files (the chain-of-custody file is not
needed by any tool). It then defines the three tools from the case brief. Each tool is an
ordinary function with `@tool` written above it. The text in triple quotes at the top of each
tool is its **description**: the AI reads it later to decide when and how to use the tool, so
it is written for the AI. Each tool returns its findings as JSON text (labelled fields), so the
AI and you get the same clear, structured result.

`inspect_image_evidence` also grades each photo against the description. For each feature
named in the description (black, SUV, roof rack) it checks the vehicle detection: all three
match with no conflicts gives `strong`, at least two matches gives `moderate`, anything less
gives `weak`.

**Output:** `Defined 3 tools: [...]` with the three tool names. The tools are kept together in
a list called `forensic_tools`. Step A.2 shows their description cards; Step A.3 runs them by
hand; Step B.1 hands the same list to the agent.

In [ ]:
# Step A.1: load the case records into memory, then build the three tools that read them.
import csv   # csv: reads spreadsheet-style files (rows of comma-separated values)
import json  # json: reads and writes structured data (lists and labelled fields)
from pathlib import Path

data_dir = Path('data')  # the folder of case files written in Step 0.3

# Helper: read one .csv table and return its rows. Each row becomes a dictionary (a list of
# name -> value pairs), for example {'file_name': 'IMG_2044.jpg', 'color': 'black', ...}.
def load_csv_records(path):
    with path.open(newline='') as h:
        return list(csv.DictReader(h))

# Read five evidence files into memory (the chain-of-custody file is not needed by any tool).
artifact_manifest = json.loads((data_dir / 'artifact_manifest.json').read_text())
media_index = load_csv_records(data_dir / 'media_index.csv')
image_metadata = load_csv_records(data_dir / 'image_metadata.csv')
vehicle_detections = load_csv_records(data_dir / 'vehicle_detections.csv')
listing_drafts = json.loads((data_dir / 'listing_drafts.json').read_text())

# Build quick look-up tables, so a tool can jump straight to one photo's records by file name.
media_by_name = {r['file_name']: r for r in media_index}
metadata_by_name = {r['file_name']: r for r in image_metadata}
detections_by_name = {r['file_name']: r for r in vehicle_detections}

# Helper: tidy a folder name. Spaces and slashes at the ends are removed, and the word
# 'gallery' is treated as the phone's camera folder, DCIM/Camera.
def normalize_root(root):
    normalized = root.strip().strip('/')
    return 'DCIM/Camera' if normalized.lower() == 'gallery' else normalized

# Helper: turn a Python result into neatly indented JSON text. Every tool returns this.
def as_json(payload):
    return json.dumps(payload, indent=2)

# ----- Tool 1: list the photos in a folder that were created on or after a given time.
# The @tool line turns the function into a tool the AI can ask to use. The text in triple
# quotes is the tool's description: the AI reads it, so it is written for the AI, not for you.
@tool
def list_media_files(root: str, date_from: str):
    """
    List media files from the requested folder on or after a given UTC timestamp.

    Args:
        root (str): Folder name such as "DCIM/Camera" or "gallery/".
        date_from (str): Lower UTC bound in ISO-8601 format.
    """
    # Keep only photos in the requested folder created on or after date_from. Times written in
    # this standard format sort correctly as plain text, so >= means "on or after".
    normalized_root = normalize_root(root)
    matches = [
        {'file_name': row['file_name'], 'folder': row['folder'],
         'created_at_utc': row['created_at_utc'], 'path': row['path'],
         'size_bytes': row['size_bytes']}
        for row in media_index
        if row['created_at_utc'] >= date_from and row['folder'].startswith(normalized_root)
    ]
    return as_json(matches)

# ----- Tool 2: look at one photo from every angle and compare it with the case description.
@tool
def inspect_image_evidence(file_name: str, case_description: str):
    """
    Inspect one candidate image by combining file details, image metadata, vehicle detections,
    and a comparison to the case vehicle description.

    Args:
        file_name (str): Image filename such as "IMG_2044.jpg".
        case_description (str): Known case description such as "black SUV with roof rack".
    """
    # Find this photo's three records: file details, image metadata, and vehicle detection.
    media_record = media_by_name.get(file_name)
    metadata_record = metadata_by_name.get(file_name)
    detection_record = detections_by_name.get(file_name)
    # If any record is missing, say so plainly instead of guessing.
    if media_record is None:
        return as_json({'error': f'No media record found for {file_name}'})
    if metadata_record is None:
        return as_json({'error': f'No metadata found for {file_name}'})
    if detection_record is None:
        return as_json({'error': f'No detection found for {file_name}'})
    # Compare the detection with the description, one feature at a time. Each feature that the
    # description mentions (black, SUV, roof rack) goes into either 'matching' or 'conflicting'.
    description = case_description.lower()
    matching, conflicting = [], []
    if 'black' in description:
        (matching if detection_record['color'].lower() == 'black' else conflicting).append(
            'black' if detection_record['color'].lower() == 'black' else f"color={detection_record['color']}")
    if 'suv' in description:
        (matching if detection_record['body_type'].lower() == 'suv' else conflicting).append(
            'SUV' if detection_record['body_type'].lower() == 'suv' else f"body_type={detection_record['body_type']}")
    if 'roof rack' in description:
        (matching if detection_record['roof_rack'].lower() == 'yes' else conflicting).append(
            'roof rack' if detection_record['roof_rack'].lower() == 'yes' else 'roof rack not detected')
    # Grade the match: 3 matches and no conflicts = strong, at least 2 = moderate, else weak.
    if len(matching) == 3 and not conflicting:
        match_strength = 'strong'
    elif len(matching) >= 2:
        match_strength = 'moderate'
    else:
        match_strength = 'weak'
    # Return everything an examiner needs to judge this photo, as JSON text.
    return as_json({
        'file_name': file_name,
        'file_record': {'path': media_record['path'], 'folder': media_record['folder'],
                        'created_at_utc': media_record['created_at_utc'], 'size_bytes': media_record['size_bytes']},
        'image_metadata': metadata_record,
        'vehicle_detection': detection_record,
        'feature_comparison': {'case_description': case_description, 'matching_features': matching,
                               'conflicting_features': conflicting, 'match_strength': match_strength},
    })

# ----- Tool 3: find the saved sale-listing drafts created on or after a given time.
@tool
def inspect_listing_records(date_from: str):
    """
    Inspect saved listing-draft records created on or after a given UTC timestamp.

    Args:
        date_from (str): Lower UTC bound in ISO-8601 format.
    """
    matches = [d for d in listing_drafts if d['created_at_utc'] >= date_from]
    return as_json(matches)

# Keep the three tools together in one list. Part A uses them by hand; Part B gives the same
# list to the agent.
forensic_tools = [list_media_files, inspect_image_evidence, inspect_listing_records]
print('Defined', len(forensic_tools), 'tools:', [t.name for t in forensic_tools])

### Step A.2: Look at the tools' description cards

**Purpose:** see each tool the way the AI will see it. In Part B, this card is **all** the AI
knows about a tool when it decides which one to call and what inputs to give it.

**Input:** `forensic_tools` from Step A.1.

**How it works:** the code takes each tool's description card (made by `@tool` in Step A.1) and
prints them all as JSON text.

**Output:** one card per tool, showing its `name`, its `description` (the text in triple quotes
from Step A.1), and its `parameters`: the inputs it needs and their type (`str` means text).
Nothing new is saved; this step is a preview. Step A.3 runs the tools.

In [ ]:
# Step A.2: print the description card of each tool. This is exactly what the AI reads in Part B.
# Each card is stored as JSON text (fn_signature); json.loads turns it back into Python data,
# and json.dumps prints it neatly indented.
tool_catalog = {t.name: json.loads(t.fn_signature) for t in forensic_tools}
print(json.dumps(tool_catalog, indent=2))

### Step A.3: Run the three tools yourself

**Purpose:** gather the evidence step by step, the way an examiner would, so you can see every
tool call and every raw result.

**Input:** the three tools from Step A.1. The vehicle description (`black SUV with roof rack`)
is typed into the cell, and the start of the time window (`2026-01-02T00:00:00Z`) is read from
the case summary file.

**How it works:** the code runs the tools in the natural order:

1. `list_media_files` finds the camera photos created in the time window.
2. `inspect_image_evidence` inspects two of them: `IMG_2044.jpg` (the strongest match) and
   `IMG_2051.jpg` (a weaker one), so you see evidence that both supports and limits the case.
3. `inspect_listing_records` finds the sale-listing drafts created in the time window.

It writes down which tools were called with which inputs (the **tool-call log**) and gathers
all the results into one evidence package.

**Output:** a short summary: how well each of the two photos matches, and the titles of the
drafts found. The full results are saved as `manual_evidence_bundle` and the log as
`manual_tool_log`. Step A.4 hands both to the AI to write up.

In [ ]:
# Step A.3: run the three tools yourself, in order, so every call and every result is visible.
# The vehicle description from the theft report, and the start of the time window under
# investigation (read from the case summary file).
case_description = 'black SUV with roof rack'
date_from = artifact_manifest['incident_window_utc']['start']

# .run(...) runs a tool directly. json.loads turns each tool's JSON reply back into Python data.
# 1. Which camera photos were created in the time window?
candidate_media = json.loads(list_media_files.run(root='DCIM/Camera', date_from=date_from))
# 2. Inspect two candidate photos against the description.
img_2044 = json.loads(inspect_image_evidence.run(file_name='IMG_2044.jpg', case_description=case_description))
img_2051 = json.loads(inspect_image_evidence.run(file_name='IMG_2051.jpg', case_description=case_description))
# 3. Which sale-listing drafts were created in the time window?
listing_results = json.loads(inspect_listing_records.run(date_from=date_from))

# The tool-call log: which tool you called, with which inputs, in order.
manual_tool_log = [
    {'tool': 'list_media_files', 'arguments': {'root': 'DCIM/Camera', 'date_from': date_from}},
    {'tool': 'inspect_image_evidence', 'arguments': {'file_name': 'IMG_2044.jpg', 'case_description': case_description}},
    {'tool': 'inspect_image_evidence', 'arguments': {'file_name': 'IMG_2051.jpg', 'case_description': case_description}},
    {'tool': 'inspect_listing_records', 'arguments': {'date_from': date_from}},
]
# All the results together in one evidence package.
manual_evidence_bundle = {
    'candidate_media': candidate_media,
    'image_evidence': {'IMG_2044.jpg': img_2044, 'IMG_2051.jpg': img_2051},
    'listing_results': listing_results,
}
# A short summary: how well each photo matches, and the titles of the drafts found.
print('IMG_2044 match:', img_2044['feature_comparison']['match_strength'])
print('IMG_2051 match:', img_2051['feature_comparison']['match_strength'])
print('drafts found:', [d['title'] for d in listing_results])

### Step A.4: Ask the AI to write the report from your evidence

**Purpose:** let the AI turn the evidence you gathered into a structured report, the way a
colleague might write up your notes.

**Input:** `manual_tool_log` and `manual_evidence_bundle` from Step A.3, plus `client` and
`MODEL` from Step 0.2.

**How it works:** this step does **not** run any tools. The code writes out the case question,
your tool-call log, the evidence, and the five parts the report must have: (1) the tool-call
log, (2) the strongest timestamp evidence, (3) the strongest vehicle-match evidence, (4) a
conclusion label (confirmed, likely, or unconfirmed) with a confidence from 0 to 1 for each
major claim, and (5) which evidence supports each claim, and its limits. It sends all of this
to the AI, with instructions to keep what the evidence shows separate from what it concludes.
The AI only has the evidence you gave it, but it can still make mistakes or overstate things,
so check its work.

**Output:** the AI's five-part report, printed as plain text and saved as `manual_report`.
Check that every claim points back to a tool result from Step A.3. The case question
(`CASE_QUESTION`) and the five-part format (`REPORT_FORMAT`) are reused in Step B.1.

In [ ]:
# Step A.4: ask the AI to write a report using ONLY the evidence you gathered in Step A.3.
# The case question. Step B.1 reuses it.
CASE_QUESTION = (
    'Does the phone contain confirmed, likely, or unconfirmed evidence that a stolen black SUV '
    'was photographed and prepared for an online sale after January 2, 2026?')

# The five parts every report must have. Step B.1 reuses this too.
REPORT_FORMAT = '''Return:
(1) tool-call log
(2) strongest timestamp evidence
(3) strongest vehicle-match evidence
(4) conclusion label (confirmed, likely, or unconfirmed) with confidence 0-1 per major claim
(5) explicit evidence mapping and limits'''

# Build the request: the question, your tool-call log, the evidence, and the report format.
# json.dumps turns the Python data back into readable text for the AI.
manual_prompt = f'''You are a careful forensic analyst. Stay inside the evidence.
Use the manually collected tool outputs below to answer the case question.

Case question:
{CASE_QUESTION}

Evidence-gathering steps:
{json.dumps(manual_tool_log, indent=2)}

Collected evidence:
{json.dumps(manual_evidence_bundle, indent=2)}

{REPORT_FORMAT}'''

# Send it to the AI with a short job description (the 'system' message), then print the report.
manual_report = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a careful forensic analyst. Distinguish observed evidence from conclusions and state unresolved limits explicitly.'},
        {'role': 'user', 'content': manual_prompt},
    ],
).choices[0].message.content
print(manual_report)

## Part B: Hand the Tools to `ToolAgent`

Now the AI gets the same three tools and chooses the calls itself. An **agent** is a program
that uses an AI model to carry out several steps toward a goal on its own. Here is the loop the
`ToolAgent` runs:

1. It shows the AI the description cards of the tools (the ones you saw in Step A.2) and the
   task.
2. The AI replies with a request to use a tool, written inside `<tool_call>` tags: the tool's
   name and the inputs to give it.
3. The agent runs that tool on the evidence and sends the result back to the AI as an
   **Observation**.
4. Steps 2 and 3 repeat until the AI writes a plain-text answer with no more tool requests.

Two safety rules are built in. If the AI tries to answer before it has used any tool, the
agent tells it to call a tool first (up to two times), because an answer with no evidence
behind it cannot be trusted. And the loop has a round limit, after which the AI must answer
from what it has.

Your job shifts from *calling* the tools to *judging* whether the agent called the right ones.

### Step B.1: Build the agent and write its task

**Purpose:** set up the agent with the same tools and the same case question as Part A, so the
two workflows can be compared fairly.

**Input:** `ToolAgent` from Step 0.4, `forensic_tools` from Step A.1, `case_description` from
Step A.3, `CASE_QUESTION` and `REPORT_FORMAT` from Step A.4, and `client` and `MODEL` from
Step 0.2.

**How it works:** the code creates the agent and writes its task: the case question, a
suggested order of tools, a few hints about the data (the folder name and the vehicle
description), and the same five-part report format. Nothing is sent to the AI yet. From now on
the agent, not you, decides which tools to call and with what inputs.

**Output:** `Agent ready with tools: [...]` with the three tool names. The agent is saved as
`tool_agent` and its task as `agent_user_msg`. Step B.2 runs them.

In [ ]:
# Step B.1: create the agent, give it the same three tools, and write its task.
tool_agent = ToolAgent(tools=forensic_tools, client=client, model=MODEL)

# The agent's task: the same case question, a suggested order of tools, a few hints about the
# data, and the same report format. The agent decides the actual tool calls itself.
agent_user_msg = f'''Use the available forensic tools to answer this case question.

Case question:
{CASE_QUESTION}

For a complete review, first use list_media_files. Inspect one or more file_name values it returns
with inspect_image_evidence, then use inspect_listing_records to examine related drafts.

Relevant dataset hints:
- the media folder is DCIM/Camera
- the known vehicle description is: {case_description}
- inspect_image_evidence combines timestamps, metadata, vehicle detection, and a feature comparison
- only use file_name values returned by list_media_files when you call inspect_image_evidence

Focus on the strongest image candidate, but mention competing evidence if it weakens the conclusion.

{REPORT_FORMAT}'''
print('Agent ready with tools:', [t.name for t in forensic_tools])

### Step B.2: Run the agent

**Purpose:** let the agent choose and run the tool calls on its own, then write its report.

**Input:** `tool_agent` and `agent_user_msg` from Step B.1.

**How it works:** the code starts the agent's loop (described in the Part B introduction) and
waits for it to finish. Every time the AI asks for a tool, the agent prints what it is doing.
This can take a little while, because each round is a separate request to the AI.

**Output:** for each tool request, a `Using Tool:` line with the tool's name, a `Tool call:`
line with the inputs used, and a `Tool result:` block with what the tool found. (A `Skipping`
line means the AI made a broken request, or asked for a tool that does not exist. A result
starting `Tool error:` means the tool could not run with the inputs the AI gave it.) Then comes
`===== AGENT REPORT =====` and the agent's final report, saved as `agent_report`.

Compare the agent's tool calls with your own from Step A.3. Did it inspect the same photos?
Did it skip a tool? Did it claim more than the tool results show? You will use this in Your
Evidence Report and Compare the Two Workflows.

In [ ]:
# Step B.2: run the agent's loop end to end. The agent decides the tool calls; you judge them.
agent_report = tool_agent.run(user_msg=agent_user_msg)
# Print a heading, then the agent's final report.
print('\n===== AGENT REPORT =====\n')
print(agent_report)

## Your Evidence Report

**Purpose:** write the forensic conclusion yourself. Your report, not either AI's, is the
forensic conclusion.

**Input:** your tool results from Step A.3, the AI's report from Step A.4, and the agent's tool
calls and report from Step B.2.

**How it works:** in the cell below, replace each `'...'` with your own words, keeping the quote
marks. For `conclusion_label`, keep just one of the three words. Replace each `0.0` with a
confidence from 0 (no confidence) to 1 (certain), with no quote marks. Then run the cell. Base
every field on the tool outputs you judge most relevant. Tip: do not use the `'` character
inside an answer (write "do not", not "don't"), or Python will stop with an error.

**Output:** the cell prints each field with your answer, so you can check it before you submit.

The five fields:

1. **Tool-call log:** the calls you would stand behind (from Part A, or the agent's, or your own).
2. **Strongest timestamp evidence:** the capture or draft time that matters most, with its source.
3. **Strongest vehicle-match evidence:** the image and features that best match the stolen vehicle.
4. **Conclusion label:** confirmed / likely / unconfirmed, with a confidence (0-1) per major claim.
5. **Evidence mapping and limits:** which tool output supports each claim, and what is **not** proven
   (for example, that the listing was ever published or the vehicle sold).

In [ ]:
# Your Evidence Report: replace each '...' with your own words and each 0.0 with a confidence
# from 0 to 1, then run this cell.
report = {
    'tool_call_log': ['...'],
    'strongest_timestamp_evidence': '...',
    'strongest_vehicle_match_evidence': '...',
    'conclusion_label': 'confirmed | likely | unconfirmed',
    'confidence_per_claim': {'photographed_stolen_suv': 0.0, 'prepared_for_sale': 0.0},
    'evidence_mapping_and_limits': '...',
}
# Print each field with your answer next to it.
for k, v in report.items():
    print(k, '->', v)

### Compare the two workflows

This part has no code. In your submission, answer briefly. You can add a text cell below this
one (in Colab: **Insert -> Text cell**) and type your answers there.

1. Did `ToolAgent` leave out a needed tool, or inspect a different image than you did?
2. Name one agent choice or conclusion you would **accept**, and one you would **correct or reject**.
3. What does the evidence **not** establish, even at its strongest?

## Submit

Save the notebook with Part A, Part B, your report, and your comparison answers filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*Case data and the lab concept are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). The tool
framework is adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*